Environment setup and imports

In [49]:
# Python + DS stack
# !pip install pandas numpy plotly tensorflow scikit-learn
import os
import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timedelta
from typing import Dict, Tuple

# 🚀 FidZulu Notebook Bootstrap
import sys
import importlib, fidzulu
importlib.reload(fidzulu)
import pandas as pd
pd.options.plotting.backend = "plotly"
import plotly.graph_objects as go

import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers # type: ignore

# FidZulu imports: FidZulu bootstrap + TensorFlow version check
# 🚀 FidZulu Notebook Bootstrap

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(project_root, "src")

# Add project root
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Add src/
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from fidzulu.db import oracle_engine
from fidzulu.repositories.price_repository import PriceRepository
from fidzulu.business.price_wrangler import PriceDataWrangler
print(tf.__version__)
# This forces TensorFlow to compile your model into a fast graph
tf.config.run_functions_eagerly(False)


2.20.0


GPU vs CPU

In [50]:
# List available devices
print("Available devices:", tf.config.list_physical_devices())

# For small tabular models, GPU overhead is larger than GPU benefit
# Thus, force TensorFlow to use CPU only
tf.config.set_visible_devices([], 'GPU')

# Or, if you want to force GPU (assuming you have one):
# gpus = tf.config.list_physical_devices('GPU')
# if gpus:
#     tf.config.set_visible_devices(gpus[0], 'GPU')

print("Now using:", tf.config.get_visible_devices())

Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Now using: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Load Raw Data

We connect to the database and retrieve raw price data for a given category.

In [51]:
# Adjust connection string to your environment
engine = oracle_engine()
repo = PriceRepository(engine)
raw_data = repo.get_prices_by_category(cat_id=13) # Vegetable category
print(type(raw_data))
print(raw_data.keys())

raw_data

2026-01-02 14:42:24,907 INFO fidzulu.config: Loaded DBConfig: host=localhost, port=1521, service=xepdb1, user=fidzulu_pythonmluser
2026-01-02 14:42:24,908 INFO fidzulu.db: Creating Oracle engine for fidzulu_pythonmluser@localhost:1521/xepdb1
2026-01-02 14:42:24,909 INFO fidzulu.db: Creating Oracle engine with DSN and user credentials
2026-01-02 14:42:24,910 INFO fidzulu.db: Oracle engine created successfully


<class 'dict'>
dict_keys(['CategoryID', 105, 106])


{'CategoryID': 13,
 105: {'prices': [Decimal('9.95'),
   -1,
   Decimal('2.5'),
   Decimal('2.65'),
   Decimal('2.4'),
   Decimal('2.75'),
   Decimal('2.55'),
   Decimal('2.85'),
   Decimal('2.6'),
   Decimal('2.95'),
   Decimal('2.7'),
   Decimal('3.05'),
   Decimal('2.8'),
   Decimal('3.15'),
   Decimal('2.95'),
   Decimal('3.3'),
   Decimal('3.05'),
   Decimal('3.4'),
   Decimal('3.15'),
   Decimal('3.5'),
   Decimal('3.25'),
   Decimal('3.6'),
   Decimal('3.35'),
   Decimal('10.15'),
   Decimal('3.45'),
   200,
   Decimal('3.8'),
   Decimal('3.55')],
  'start_dates': [datetime.datetime(2022, 11, 1, 0, 0),
   datetime.datetime(2022, 12, 1, 0, 0),
   datetime.datetime(2023, 1, 1, 0, 0),
   datetime.datetime(2023, 2, 1, 0, 0),
   datetime.datetime(2023, 3, 1, 0, 0),
   datetime.datetime(2023, 4, 1, 0, 0),
   datetime.datetime(2023, 5, 1, 0, 0),
   datetime.datetime(2023, 6, 1, 0, 0),
   datetime.datetime(2023, 7, 1, 0, 0),
   datetime.datetime(2023, 8, 1, 0, 0),
   datetime.datetime(2